In [5]:
import pickle
import numpy as np
import joblib

In [29]:

# Let's say you have a spectrum you want to classify.

# With an example below (wavelength vector and counts vector):
particle_types = ['Er0p5Tm02Yb97p5', 'Er01Tm02Yb20_NEW_PMAO', 'Er01Tm05Yb94_NEW_PMAO', 'Er02Tm01Yb97_NEW_PMAO',
                  'Er02Tm05Yb93', 'Er02Yb20', 'Er04Yb13', 'Tm01Yb99', 'Tm02Yb98', 'Tm10Yb90']

full_dict = {}
for particle_type in particle_types:
    with open(f"database/{particle_type}_spectra_cleaned_background substracted_should be singles I hope_1.pkl", "rb") as f:   # "rb" = read binary
        data_ptcl = pickle.load(f)
        full_dict[particle_type] = data_ptcl

# let's extract the spectrum of some random particle
particle_id = '1_1'
frame_avg = 5
particle_type = 'Er04Yb13'
particle_dict = full_dict[particle_type][particle_id]
spectrum = particle_dict[frame_avg]
print(spectrum)

[array([434.28677053, 435.65920495, 437.06111424, 438.49313141,
       439.95590307, 441.45008971, 442.976366  , 444.53542113,
       446.12795905, 447.75469885, 449.41637507, 451.11373801,
       452.8475541 , 454.61860621, 456.42769403, 458.27563444,
       460.16326184, 462.09142857, 464.06100527, 466.07288125,
       468.12796498, 470.22718437, 472.37148732, 474.56184204,
       476.79923756, 479.08468415, 481.41921376, 483.80388053,
       486.23976121, 488.72795568, 491.26958747, 493.8658042 ,
       496.51777817, 499.22670682, 501.99381335, 504.8203472 ,
       507.70758464, 510.65682938, 513.66941309, 516.74669608,
       519.89006784, 523.10094772, 526.38078555, 529.73106228,
       533.1532907 , 536.64901605, 540.21981679, 543.86730525,
       547.59312842, 551.39896862, 555.28654435, 559.25761096,
       563.31396156, 567.45742771, 571.68988035, 576.01323058,
       580.42943054, 584.94047431, 589.54839879, 594.25528462,
       599.06325712, 603.97448728, 608.99119268, 614.1

Now, we need to first process this spectrum before we feed it into the model. The function below should encompass all this processing

In [ ]:
def process_spectrum_for_model(ptcl_spectrum): # ptcl_spectrum is a list of two numpy arrays (wavelengths and counts)
    # Let's first create a grid to project the spectrum onto. 
    # We will project onto a uniform grid from 443 nm to 884 nm (that is what the model takes in)
    min_wavelength, max_wavelength = 443, 884
    unif_grid = np.linspace(min_wavelength, max_wavelength, max_wavelength - min_wavelength + 1)

    # Let's now project the spectrum onto the uniform grid
    wavelengths = ptcl_spectrum[0]
    counts = ptcl_spectrum[1]

    # project the counts onto the two wavelength grids: uniform grid and scaled grid
    counts_unif_grid = np.interp(unif_grid, wavelengths, counts) 

    # NOTE: THE ABOVE LINE COULD THROW AN ERROR IF THE GIVEN SPECTRA DOES NOT HAVE WAVELENGTHS
    # BELOW 443 AND ABOVE 884 (interpolation is not possible in that case)

    # normalize these counts
    norm_counts_unif_grid = counts_unif_grid / np.trapezoid(counts_unif_grid, unif_grid)

    # this is what we feed into the model (almost)! (without reshaping yet)
    return np.array(norm_counts_unif_grid)

In [ ]:
# Let's import the specific model we want to use (based on how many frames we're averaging over)
def import_model(frame_avg):
    trained_model = joblib.load(f"trained_models/{frame_avg}frame_classifier.pkl")
    return trained_model

In [36]:
trained_model = import_model(5)

In [37]:
# Now, we can use the model to predict the composition of our specific spectrum

# need to reshape to feed into model
processed_spectrum = process_spectrum_for_model(spectrum).reshape(1, -1)
encoded_prediction = trained_model.predict(processed_spectrum)

# Use label_encoder to extract which particle name corresponds to the integer it predicted
le = joblib.load("trained_models/label_encoder.pkl")
prediction = le.inverse_transform(encoded_prediction)

print(prediction)

['Er04Yb13']


We can see that in this case, this was the correct prediction! Yay.

In [52]:
# We can put this whole pipeline into a larger function that does everything for you.
# We'll allow the input of several spectra (a list of spectra) to predict several at a time.

# **This function can only be applied to a list of spectra averaged over the same number of frames**
def predict_spectra_full_pipeline(list_of_spectra, frame_avg): # list of spectra, where each spectrum is a list of two numpy arrays (wavelengths and counts)

    this_trained_model = import_model(frame_avg)
    label_encoder = joblib.load("trained_models/label_encoder.pkl")
    
    num_spectra = len(list_of_spectra)

    # create empty numpy array to fill in
    processed_spectra_array = np.zeros((num_spectra, 884 - 443 + 1))

    for spectrum_index in range(num_spectra):
        processed_spectrum = process_spectrum_for_model(list_of_spectra[spectrum_index])
        processed_spectra_array[spectrum_index] = processed_spectrum
    
    encoded_predictions = this_trained_model.predict(processed_spectra_array)
    predictions = label_encoder.inverse_transform(encoded_predictions)

    return predictions

In [58]:
# Example use for predicting several spectra

# Random example spectra
# let's extract the spectrum of some random particle
frame_avg = 1

particle_dict1 = full_dict['Er04Yb13']['1_1']
spectrum1 = particle_dict1[frame_avg]

particle_dict2 = full_dict['Tm10Yb90']['1_1']
spectrum2 = particle_dict2[frame_avg]

print(predict_spectra_full_pipeline([spectrum1, spectrum2], frame_avg))

['Er04Yb13' 'Tm10Yb90']


Yay it works!